In [1]:
import torch
import torchaudio
import torchaudio.transforms as T
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import os
import glob
import random
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
# --- Configuration ---
SAMPLE_RATE = 16000
CLIP_DURATION_SAMPLES = 16000  # 1 second
NUM_MFCC = 10
# --- Configuration ---
DATA_DIR = '/kaggle/input/datasets/sylkaladin/speech-commands-v2'
TARGET_WORDS = ['yes', 'no', 'zero', 'one', 'two', 
                'three', 'left', 'right', 'up', 'down']
# We append 'unknown' and 'silence' to the end
CLASSES = TARGET_WORDS + ['unknown', 'silence']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

NUM_CLASSES = len(CLASSES)

In [3]:
import hashlib
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
import torchaudio
import torchaudio.transforms as T

def get_speaker_hash(file_path):
    """
    Extracts speaker identifier from filenames (Google Speech Commands format:
    'speakerID_nohash_index.wav'). Falls back to the filename stem if not matching.
    """
    filename = os.path.basename(file_path)
    if '_nohash_' in filename:
        return filename.split('_nohash_')[0]
    return filename.split('.')[0]


def is_in_split(identifier, val_percentage=10.0, test_percentage=10.0):
    """
    Deterministically maps a speaker hash to 'train', 'val', or 'test'.
    Ensures 0% speaker overlap across splits without data leakage.
    """
    # Deterministic 32-bit integer hash from the speaker ID string
    hash_val = int(hashlib.sha1(identifier.encode('utf-8')).hexdigest(), 16)
    percent = (hash_val % 10000) / 100.0  # Range: [0.00, 99.99]

    if percent < test_percentage:
        return 'test'
    elif percent < (test_percentage + val_percentage):
        return 'val'
    else:
        return 'train'


def prepare_dataset_splits_leak_free(data_dir, target_words, classes, class_to_idx, 
                                     val_pct=10.0, test_pct=10.0):
    """
    Scans directory, splits deterministically by SPEAKER to eliminate leakage,
    and balances 'unknown' and 'silence' classes within each individual split.
    """
    # Temporary containers: split -> class_name -> list of file paths
    splits = {'train': {cls: [] for cls in classes},
              'val':   {cls: [] for cls in classes},
              'test':  {cls: [] for cls in classes}}

    print(f"Scanning and splitting dataset from: {data_dir}")
    
    for folder in os.listdir(data_dir):
        folder_path = os.path.join(data_dir, folder)
        if not os.path.isdir(folder_path) or folder == '_background_noise_':
            continue

        wav_files = glob.glob(os.path.join(folder_path, '*.wav'))
        cls_name = folder if folder in target_words else 'unknown'

        for f in wav_files:
            speaker_id = get_speaker_hash(f)
            split_bucket = is_in_split(speaker_id, val_pct, test_pct)
            splits[split_bucket][cls_name].append(f)

    # Assemble balanced datasets for each split
    datasets = {}
    for split_name in ['train', 'val', 'test']:
        sp_data = splits[split_name]
        
        # Calculate target average for this specific split
        target_counts = [len(sp_data[w]) for w in target_words if len(sp_data[w]) > 0]
        avg_target_count = int(sum(target_counts) / max(len(target_counts), 1))

        # Balance unknown
        random.seed(42)
        random.shuffle(sp_data['unknown'])
        sp_data['unknown'] = sp_data['unknown'][:avg_target_count]

        # Balance silence with dummy path tokens
        sp_data['silence'] = ["SILENCE_DUMMY_PATH"] * avg_target_count

        # Flatten into final X (paths) and y (integer label tensors)
        all_files, all_labels = [], []
        for cls in classes:
            files = sp_data[cls]
            all_files.extend(files)
            all_labels.extend([class_to_idx[cls]] * len(files))

        datasets[split_name] = (all_files, all_labels)
        print(f"[{split_name.upper()}] Total: {len(all_files)} samples | Balanced target size: {avg_target_count}")

    return datasets['train'], datasets['val'], datasets['test']

# Run the preparation
(X_train, y_train), (X_val, y_val), (X_test, y_test) = prepare_dataset_splits_leak_free(DATA_DIR,
                                                                                        TARGET_WORDS,
                                                                                        CLASSES,
                                                                                        CLASS_TO_IDX)

Scanning and splitting dataset from: /kaggle/input/datasets/sylkaladin/speech-commands-v2
[TRAIN] Total: 37737 samples | Balanced target size: 3144
[VAL] Total: 4839 samples | Balanced target size: 403
[TEST] Total: 3925 samples | Balanced target size: 327


In [4]:
def load_background_noises(data_dir):
    """Loads all long background noise files into memory for fast slicing."""
    noise_dir = os.path.join(data_dir, '_background_noise_')
    noise_files = glob.glob(os.path.join(noise_dir, '*.wav'))
    
    bg_noises = []
    for f in noise_files:
        waveform, sr = torchaudio.load(f)
        if sr != SAMPLE_RATE:
            waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)
        if waveform.shape[0] > 1: # Convert to mono
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        bg_noises.append(waveform)
    
    print(f"Loaded {len(bg_noises)} background noise files.")
    return bg_noises

In [5]:
class KeywordSpottingDataset(Dataset):
    def __init__(self, file_paths, labels, bg_noises, is_training=True):
        self.file_paths = file_paths
        self.labels = labels
        self.bg_noises = bg_noises
        self.is_training = is_training

        # MicroSpeech standard MFCC extraction: 30ms window, 20ms step, 10 coefficients
        self.mfcc_transform = T.MFCC(
            sample_rate=SAMPLE_RATE,
            n_mfcc=NUM_MFCC,
            melkwargs={"n_fft": 480, "hop_length": 320, "n_mels": 40, "center": False}
        )

    def __len__(self):
        return len(self.file_paths)

    def _get_random_noise_slice(self):
        if not self.bg_noises:
            return torch.zeros((1, CLIP_DURATION_SAMPLES))

        noise_tensor = random.choice(self.bg_noises)
        max_start = noise_tensor.shape[1] - CLIP_DURATION_SAMPLES
        if max_start <= 0:
            return F.pad(noise_tensor, (0, abs(max_start)))

        start_idx = random.randint(0, max_start)
        return noise_tensor[:, start_idx: start_idx + CLIP_DURATION_SAMPLES]

    def _pad_or_trim(self, waveform):
        seq_len = waveform.shape[1]
        if seq_len < CLIP_DURATION_SAMPLES:
            return F.pad(waveform, (0, CLIP_DURATION_SAMPLES - seq_len))
        elif seq_len > CLIP_DURATION_SAMPLES:
            return waveform[:, :CLIP_DURATION_SAMPLES]
        return waveform

    def _clean_time_shift(self, waveform, max_shift=4000):
        """
        Shifts waveform linearly and zero-pads the exposed edge.
        Avoids the audio pop/click artifacts caused by torch.roll circular wrapping.
        """
        shift = random.randint(-max_shift, max_shift)
        if shift == 0:
            return waveform

        if shift > 0:
            # Shift right: truncate end, pad front with zeros
            shifted = waveform[:, :-shift]
            return F.pad(shifted, (shift, 0))
        else:
            # Shift left: truncate start, pad back with zeros
            shift_abs = abs(shift)
            shifted = waveform[:, shift_abs:]
            return F.pad(shifted, (0, shift_abs))

    def __getitem__(self, idx):
        file_path = self.file_paths[idx]
        label = self.labels[idx]

        # 1. Load Audio
        if file_path == "SILENCE_DUMMY_PATH":
            waveform = self._get_random_noise_slice()
            waveform = waveform * random.uniform(0.1, 0.8)
        else:
            waveform, sr = torchaudio.load(file_path)
            if sr != SAMPLE_RATE:
                waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)
            if waveform.shape[0] > 1:
                waveform = torch.mean(waveform, dim=0, keepdim=True)

            waveform = self._pad_or_trim(waveform)

            # 2. Augmentations (Train only)
            if self.is_training:
                # Clean zero-padded time shift (no roll artifacts)
                waveform = self._clean_time_shift(waveform, max_shift=4000)

                # Additive background noise injection
                if random.random() < 0.8:
                    noise = self._get_random_noise_slice()
                    waveform = waveform + (noise * random.uniform(0.0, 0.2))

            waveform = torch.clamp(waveform, -1.0, 1.0)

        # 3. Feature Extraction -> [1, 10, 49]
        mfcc = self.mfcc_transform(waveform)

        return mfcc, torch.tensor(label, dtype=torch.long)

In [6]:
# 1. Load the long noise files into RAM once
bg_noises = load_background_noises(DATA_DIR)

# 2. Create Datasets
train_dataset = KeywordSpottingDataset(X_train, y_train, bg_noises, is_training=True)
val_dataset   = KeywordSpottingDataset(X_val, y_val, bg_noises, is_training=False)

test_dataset = KeywordSpottingDataset(X_test, y_test, bg_noises, is_training=False)

# 3. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4, drop_last=True)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4)
test_loader   = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4)

# Test the pipeline
for mfcc_batch, labels_batch in train_loader:
    print(f"Batch MFCC shape: {mfcc_batch.shape}") # Should be [64, 1, 10, 49]
    print(f"Batch Labels shape: {labels_batch.shape}") # Should be [64]
    break

Loaded 6 background noise files.
Batch MFCC shape: torch.Size([64, 1, 10, 49])
Batch Labels shape: torch.Size([64])


In [7]:
import torch.nn as nn

class DSCNNBlock(nn.Module):
    """A standard Depthwise Separable Convolution block."""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        # 1. Depthwise Convolution (groups = in_channels)
        self.depthwise = nn.Conv2d(
            in_channels, in_channels, kernel_size=3, 
            stride=stride, padding=1, groups=in_channels, bias=False
        )
        self.bn1 = nn.BatchNorm2d(in_channels)
        
        # 2. Pointwise Convolution (1x1 standard conv)
        self.pointwise = nn.Conv2d(
            in_channels, out_channels, kernel_size=1, 
            stride=1, padding=0, bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.bn1(x)
        x = self.relu(x)
        
        x = self.pointwise(x)
        x = self.bn2(x)
        x = self.relu(x)
        return x

class MicroSpeechDSCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        
        # Initial standard convolution to extract base features
        # Input shape: [Batch, 1, 10 (freq), 49 (time)]
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu = nn.ReLU(inplace=True)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Stack 5 DS-CNN Layers (Highly parameter efficient, fits in 12KB-64KB SRAM)
        self.ds_conv1 = DSCNNBlock(32, 64)
        self.ds_conv2 = DSCNNBlock(64, 32)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.ds_conv3 = DSCNNBlock(32, 32)
        self.ds_conv4 = DSCNNBlock(32, 16)
        
        # The Final block outputs EXACTLY 8 channels (as requested by the document)
        self.ds_conv5 = DSCNNBlock(16, 8)
        
        # Global Average Pooling (Solves the "Fatal Flaw" - ensures Temporal/Shift Invariance)
        # Squeezes spatial dimensions (H, W) down to 1x1.
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        
        # Final Dense Layer
        # Since GAP outputs 8 values, and we have 4 classes: 8 * 4 = 32 weights
        self.classifier = nn.Linear(8, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.pool1(x)
        
        x = self.ds_conv1(x)
        x = self.ds_conv2(x)
        x = self.pool2(x)
        
        x = self.ds_conv3(x)
        x = self.ds_conv4(x)
        x = self.ds_conv5(x)
        
        x = self.gap(x)
        
        # Flatten the (Batch, 8, 1, 1) output to (Batch, 8)
        x = torch.flatten(x, 1) 
        
        x = self.classifier(x)
        return x

# --- Instantiate and Test ---
model = MicroSpeechDSCNN(num_classes=NUM_CLASSES)

# Dummy input representing a batch of 1 KWS MFCC tensor [Batch, Channel, Freq, Time]
dummy_input = torch.randn(1, 1, NUM_MFCC, 49) 
output = model(dummy_input)

print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {output.shape} (Matches 4 output classes)")

Input shape: torch.Size([1, 1, 10, 49])
Output shape: torch.Size([1, 12]) (Matches 4 output classes)


In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import time
import copy

# --- Device Configuration ---
# Automatically select CUDA (Nvidia), MPS (Apple Silicon M2), or CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")


def train_epoch(model, dataloader, criterion, optimizer, device):
    """Trains the model for one epoch."""
    model.train() # Set model to training mode (enables BatchNorm tracking)
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for inputs, labels in dataloader:
        # Move data to the active device (GPU/MPS/CPU)
        inputs, labels = inputs.to(device), labels.to(device)

        # 1. Zero the gradients
        optimizer.zero_grad()

        # 2. Forward pass
        outputs = model(inputs)
        
        # 3. Calculate Loss
        loss = criterion(outputs, labels)

        # 4. Backward pass & optimize
        loss.backward()
        optimizer.step()

        # Track metrics
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct_predictions += torch.sum(preds == labels.data)
        total_samples += labels.size(0)

    epoch_loss = running_loss / total_samples
    epoch_acc = correct_predictions.double() / total_samples
    return epoch_loss, epoch_acc.item()


def evaluate(model, dataloader, criterion, device):
    """Evaluates the model on validation or test sets."""
    model.eval() # Set model to evaluation mode (freezes BatchNorm)
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    # Disable gradient calculation for inference memory efficiency
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct_predictions += torch.sum(preds == labels.data)
            total_samples += labels.size(0)

    epoch_loss = running_loss / total_samples
    epoch_acc = correct_predictions.double() / total_samples
    return epoch_loss, epoch_acc.item()

Using device: cuda


In [9]:
def main():
    # 1. Initialize the Model (From previous step)
    # Make sure MicroSpeechDSCNN is defined in your script
    model = MicroSpeechDSCNN(num_classes=NUM_CLASSES).to(device)
    
    # 2. Define Loss and Optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    
    # 3. Learning Rate Scheduler (Helps converge to a better accuracy)
    # Reduces learning rate when validation loss stops improving
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.1, patience=5
    )

    # Hyperparameters
    EPOCHS = 50
    best_val_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())

    print("-" * 50)
    print("Starting Training Loop...")
    print("-" * 50)

    for epoch in range(EPOCHS):
        start_time = time.time()

        # Train & Validate
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        # Step the scheduler based on validation loss
        scheduler.step(val_loss)

        epoch_time = time.time() - start_time

        print(f"Epoch {epoch+1:02d}/{EPOCHS} [{epoch_time:.1f}s] | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

        # Save the best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), "best_kws_dscnn.pth")
            print(f"  --> Saved new best model with Val Acc: {best_val_acc:.4f}")

    print("-" * 50)
    print(f"Training Complete. Best Validation Accuracy: {best_val_acc:.4f}")
    
    # Load the best weights into the model for final testing
    model.load_state_dict(best_model_wts)
    
    # --- Final Test Set Evaluation ---
    # Assuming you created a test_loader the same way as val_loader:
    # test_dataset = KeywordSpottingDataset(X_test, y_test, bg_noises, is_training=False)
    # test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    
    print("Running evaluation on the unseen Test Set...")
    test_loss, test_acc = evaluate(model, val_loader, criterion, device)
    print(f"Final Test Loss: {test_loss:.4f} | Final Test Acc: {test_acc:.4f}")

# Execute the main function
if __name__ == "__main__":
    main()

--------------------------------------------------
Starting Training Loop...
--------------------------------------------------
Epoch 01/50 [113.6s] | Train Loss: 1.8306 Acc: 0.4247 | Val Loss: 1.1424 Acc: 0.6811
  --> Saved new best model with Val Acc: 0.6811
Epoch 02/50 [67.1s] | Train Loss: 0.9884 Acc: 0.7146 | Val Loss: 0.6252 Acc: 0.8237
  --> Saved new best model with Val Acc: 0.8237
Epoch 03/50 [66.8s] | Train Loss: 0.6974 Acc: 0.7880 | Val Loss: 0.4981 Acc: 0.8524
  --> Saved new best model with Val Acc: 0.8524
Epoch 04/50 [67.9s] | Train Loss: 0.5910 Acc: 0.8156 | Val Loss: 0.4376 Acc: 0.8698
  --> Saved new best model with Val Acc: 0.8698
Epoch 05/50 [66.8s] | Train Loss: 0.5398 Acc: 0.8311 | Val Loss: 0.3964 Acc: 0.8783
  --> Saved new best model with Val Acc: 0.8783
Epoch 06/50 [66.7s] | Train Loss: 0.5182 Acc: 0.8370 | Val Loss: 0.3719 Acc: 0.8872
  --> Saved new best model with Val Acc: 0.8872
Epoch 07/50 [66.7s] | Train Loss: 0.4901 Acc: 0.8448 | Val Loss: 0.3744 Acc: 0.